[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/03_ONNX_Architecture_and_Internals/01_Computation_Graph_Basics/Computation_Graph_Basics_Deep_Dive.ipynb)

# 3.1 Computation Graph Basics — Deep Dive

ONNX uses a **directed acyclic graph (DAG)** to represent computation. This section establishes the formal mathematical foundations: DAG theory, topological ordering, forward pass execution, and the SSA naming convention.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [What is a Computation Graph?](#section-1) | Informal and formal definitions |
| 2 | [DAG Formal Definition](#section-2) | $G = (V, E)$, acyclicity, topological sort |
| 3 | [ONNX Graph Components](#section-3) | Nodes, edges (implicit), inputs, outputs, initializers |
| 4 | [SSA Naming and Edge Wiring](#section-4) | Single Static Assignment discipline |
| 5 | [Topological Execution](#section-5) | Forward pass algorithm |
| 6 | [Building and Visualizing a Graph](#section-6) | Code example with diagram |
| 7 | [Numerical Forward Pass](#section-7) | Tracing values through the graph |
| 8 | [Static vs Dynamic Shapes](#section-8) | Shape semantics and trade-offs |
| 9 | [Key Takeaways & Interview Questions](#section-9) | Summary |

### Prerequisites

- Completed Module 1 (ONNX with Python) and Module 2 (Introduction to ONNX)
- Basic graph theory vocabulary (nodes, edges, directed)

<a id='section-1'></a>
## Section 1: What is a Computation Graph?

### Informal Definition

A computation graph is a structured description of a mathematical function as a **composition of elementary operations**. Instead of writing a single formula like $y = \text{ReLU}(Wx + b)$, we decompose it into individual steps, each represented as a node in a graph:

```
Formula: y = ReLU(Wx + b)        Computation Graph:

                                  W ──▶ ┌────────┐
                                        │ MatMul │──▶ h
                                  x ──▶ └────────┘
                                                      ╲
                                                       ▼
                                  b ──────────▶ ┌─────┐
                                                │ Add │──▶ z
                                                └─────┘
                                                          ╲
                                                           ▼
                                                    ┌──────┐
                                                    │ Relu │──▶ y
                                                    └──────┘
```

### Why Graphs?

| Property | Benefit |
|----------|--------|
| **Explicit data flow** | Every intermediate value is named and traceable |
| **Scheduling** | Topological ordering yields a valid execution order |
| **Optimization** | Compilers can fuse nodes, eliminate dead code, reorder operations |
| **Parallelism** | Independent branches can execute simultaneously |
| **Analysis** | Shape and type inference propagate along edges |
| **Portability** | The graph is a platform-independent representation |

### ONNX Graph Visualization

![ONNX Computation Graph](assets/onnx_computation_graph.png)

<a id='section-2'></a>
## Section 2: DAG Formal Definition

### Definition 2.1 (Directed Acyclic Graph)

An ONNX computation graph is formally a **DAG** $G = (V, E)$ where:

$$V = \{v_1, v_2, \ldots, v_n\} \quad \text{(operator nodes)}$$
$$E = \{(v_i, v_j) \mid v_i \text{ produces a tensor consumed by } v_j\} \quad \text{(data flow edges)}$$

### Definition 2.2 (Acyclicity)

The graph must have **no directed cycles**: there is no sequence $v_{i_1}, v_{i_2}, \ldots, v_{i_k}$ such that:

$$(v_{i_1}, v_{i_2}) \in E, \; (v_{i_2}, v_{i_3}) \in E, \; \ldots, \; (v_{i_k}, v_{i_1}) \in E$$

### Definition 2.3 (Topological Ordering)

The acyclicity constraint guarantees the existence of a **topological ordering** $\sigma: V \to \{1, 2, \ldots, n\}$ such that:

$$\forall (v_i, v_j) \in E: \sigma(v_i) < \sigma(v_j)$$

This ordering ensures that every node's inputs are computed before the node itself executes.

### Theorem 2.1 (Existence of Topological Sort)

A directed graph $G$ admits a topological ordering **if and only if** $G$ is acyclic (is a DAG).

**Proof sketch**: ($\Rightarrow$) If a topological ordering exists, any cycle would require $\sigma(v) < \sigma(v)$, a contradiction. ($\Leftarrow$) Every DAG has at least one vertex with in-degree 0 (a "source"); repeatedly selecting and removing sources produces a valid ordering.

### Kahn's Algorithm (Topological Sort)

```
TOPOLOGICAL_SORT(G):
  1. Compute in-degree for each node
  2. Queue ← all nodes with in-degree 0
  3. While Queue is not empty:
     a. v ← dequeue
     b. Append v to result
     c. For each successor u of v:
        - Decrement in-degree(u)
        - If in-degree(u) == 0: enqueue u
  4. If result size < |V|: ERROR (graph has a cycle)
```

Complexity: $O(|V| + |E|)$

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install onnx onnxruntime numpy matplotlib

<a id='section-3'></a>
## Section 3: ONNX Graph Components

### The Five Component Types

| Component | Protobuf Type | Role | Count |
|-----------|:-------------|------|:-----:|
| **Inputs** | `ValueInfoProto` | Declare typed tensor interfaces for runtime data | $\geq 1$ |
| **Outputs** | `ValueInfoProto` | Declare what the graph produces | $\geq 1$ |
| **Nodes** | `NodeProto` | Execute operators on named tensors | $\geq 0$ |
| **Initializers** | `TensorProto` | Constant weights stored in the model | $\geq 0$ |
| **Edges** | *(implicit)* | Connect nodes via SSA name matching | automatic |

### The Complete Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                      ModelProto                              │
│  ┌───────────────────────────────────────────────────────┐  │
│  │                    GraphProto                          │  │
│  │                                                       │  │
│  │  ┌──────────────┐  ┌──────────────────────────────┐  │  │
│  │  │   INPUTS     │  │      INITIALIZERS            │  │  │
│  │  │ (ValueInfo)  │  │     (TensorProto)            │  │  │
│  │  │ ┌──┐         │  │  ┌──┐ ┌──┐                   │  │  │
│  │  │ │X │ (N,D)   │  │  │W │ │b │  stored data     │  │  │
│  │  │ └┬─┘         │  │  └┬─┘ └┬─┘                   │  │  │
│  │  └──┼───────────┘  └──┼────┼──────────────────────┘  │  │
│  │     │                  │    │                          │  │
│  │     ▼                  ▼    │     NODES (NodeProto)   │  │
│  │  ┌───────────────────────┐  │                          │  │
│  │  │ MatMul(X, W) → XW    │  │                          │  │
│  │  └─────────┬─────────────┘  │                          │  │
│  │            │                 ▼                          │  │
│  │            ▼    ┌────────────────┐                     │  │
│  │         ┌───────────────────────┐│                     │  │
│  │         │ Add(XW, b) → pre_act ││                     │  │
│  │         └─────────┬─────────────┘│                     │  │
│  │                   │              │                      │  │
│  │                   ▼              │                      │  │
│  │         ┌──────────────────┐     │                     │  │
│  │         │ Relu(pre_act) → Y│     │                     │  │
│  │         └────────┬─────────┘     │                     │  │
│  │                  │                                      │  │
│  │                  ▼                                      │  │
│  │  ┌──────────────────┐                                  │  │
│  │  │    OUTPUTS       │                                  │  │
│  │  │   (ValueInfo)    │                                  │  │
│  │  │   Y  (N,K)       │                                  │  │
│  │  └──────────────────┘                                  │  │
│  └───────────────────────────────────────────────────────┘  │
│  opset_import: [ai.onnx v18]                                │
└─────────────────────────────────────────────────────────────┘
```

<a id='section-4'></a>
## Section 4: SSA Naming and Edge Wiring

### Single Static Assignment (SSA)

ONNX uses **SSA discipline**: each tensor name is produced by **exactly one** source (either a graph input, an initializer, or a node output).

### Edge Creation Rule

If node $A$ produces output `"h1"` and node $B$ lists `"h1"` as an input, there is an implicit edge $A \to B$:

$$\text{Edge}(A, B) \iff \exists\, t: t \in \text{outputs}(A) \wedge t \in \text{inputs}(B)$$

### Fan-Out (One-to-Many)

A single tensor name can be consumed by **multiple** downstream nodes:

```
        ┌─────┐
  h ───▶│ Add │
  │     └─────┘
  │     ┌─────┐
  └────▶│ Mul │    ← h consumed by two nodes (fan-out)
        └─────┘
```

### Consequences

1. **Names must be unique**: No two nodes can produce the same output name
2. **Names must be defined before use**: Topological order ensures this
3. **Dead code is possible**: If a node's output is never consumed, it is dead

<a id='section-5'></a>
## Section 5: Topological Execution — The Forward Pass

### The Execution Algorithm

The forward pass evaluates the graph in topological order:

$$\text{For each node } v_i \text{ in topological order } \sigma:$$
$$\quad \text{outputs}(v_i) = f_{\text{op}}(\text{inputs}(v_i), \text{attributes}(v_i))$$

### Algorithm

```
FORWARD_PASS(G, feed_dict):
  1. env ← {}  (tensor name → value mapping)
  2. For each graph input i:
       env[i.name] ← feed_dict[i.name]
  3. For each initializer w:
       if w.name not in env:  (don't override feed_dict)
         env[w.name] ← w.data
  4. For each node v in topological order:
       input_values ← [env[name] for name in v.input]
       output_values ← EXECUTE_OP(v.op_type, input_values, v.attributes)
       for name, value in zip(v.output, output_values):
         env[name] ← value
  5. Return [env[name] for name in graph.output]
```

### Complexity

The forward pass visits each node exactly once, so the graph-level overhead is $O(|V| + |E|)$. The total runtime is dominated by the operator computations:

$$T_{\text{total}} = \sum_{v \in V} T_{\text{op}}(v)$$

For a linear regression with MatMul$(N \times D, D \times K)$ and Add:

$$T_{\text{total}} \approx O(NDK) + O(NK) \approx O(NDK)$$

<a id='section-6'></a>
## Section 6: Building and Visualizing a Graph

### Two-Layer MLP

Let's build and visualize a two-layer MLP: $Y = W_2 \cdot \text{ReLU}(W_1 \cdot X + b_1) + b_2$

### Computation Graph

![Linear Regression Graph](assets/dot_linreg.png)

![Linear Regression with Initializers](assets/dot_linreg2.png)

In [ ]:
import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper
from onnx.checker import check_model
import onnxruntime as ort

batch, in_feat, hidden, out_feat = 1, 4, 5, 3

rng = np.random.default_rng(0)
W1 = rng.standard_normal((in_feat, hidden)).astype(np.float32)
b1 = rng.standard_normal((hidden,)).astype(np.float32)
W2 = rng.standard_normal((hidden, out_feat)).astype(np.float32)
b2 = rng.standard_normal((out_feat,)).astype(np.float32)

# Create initializers
init_w1 = numpy_helper.from_array(W1, name='W1')
init_b1 = numpy_helper.from_array(b1, name='b1')
init_w2 = numpy_helper.from_array(W2, name='W2')
init_b2 = numpy_helper.from_array(b2, name='b2')

# Define I/O
X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [batch, in_feat])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [batch, out_feat])

# Define computation nodes
nodes = [
    helper.make_node('MatMul', ['X', 'W1'], ['pre1'], name='fc1'),
    helper.make_node('Add', ['pre1', 'b1'], ['act_in'], name='bias1'),
    helper.make_node('Relu', ['act_in'], ['h'], name='relu1'),
    helper.make_node('MatMul', ['h', 'W2'], ['pre2'], name='fc2'),
    helper.make_node('Add', ['pre2', 'b2'], ['Y'], name='bias2'),
]

graph = helper.make_graph(
    nodes, 'two_layer_mlp', [X], [Y],
    initializer=[init_w1, init_b1, init_w2, init_b2])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])
check_model(model)

print('Two-Layer MLP Graph')
print('=' * 60)
print(f'Nodes: {len(model.graph.node)}')
print(f'Initializers: {len(model.graph.initializer)}')
print(f'Inputs: {[i.name for i in model.graph.input]}')
print(f'Outputs: {[o.name for o in model.graph.output]}')
print(f'\nComputation Flow:')
for i, n in enumerate(model.graph.node):
    print(f'  [{i}] {n.name:8s} {n.op_type:8s} {list(n.input):25s} → {list(n.output)}')

In [ ]:
# Build the explicit edge map
producers = {}
consumers = {}

for inp in model.graph.input:
    producers[inp.name] = ('INPUT', inp.name)
for init in model.graph.initializer:
    producers[init.name] = ('INIT', init.name)
for i, node in enumerate(model.graph.node):
    for out_name in node.output:
        producers[out_name] = (f'node[{i}]', node.op_type)
    for in_name in node.input:
        consumers.setdefault(in_name, []).append((f'node[{i}]', node.op_type))
for out in model.graph.output:
    consumers.setdefault(out.name, []).append(('OUTPUT', out.name))

print('Edge Map (tensor: producer → consumers):')
print('-' * 60)
all_names = sorted(set(list(producers.keys()) + list(consumers.keys())))
for name in all_names:
    prod = producers.get(name, ('?', '?'))
    cons = consumers.get(name, [])
    cons_str = ', '.join(f'{c[1]}' for c in cons) or '(unused)'
    print(f'  {name:10s}: {prod[1]:8s} → {cons_str}')

<a id='section-7'></a>
## Section 7: Numerical Forward Pass

In [ ]:
# Run inference and trace values
sess = ort.InferenceSession(model.SerializeToString(),
                            providers=['CPUExecutionProvider'])
x = rng.standard_normal((batch, in_feat)).astype(np.float32)
ort_result = sess.run(None, {'X': x})[0]

# Manual forward pass
env = {'X': x, 'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

steps = [
    ('MatMul(X, W1)', 'pre1', lambda: env['X'] @ env['W1']),
    ('Add(pre1, b1)', 'act_in', lambda: env['pre1'] + env['b1']),
    ('Relu(act_in)', 'h', lambda: np.maximum(env['act_in'], 0)),
    ('MatMul(h, W2)', 'pre2', lambda: env['h'] @ env['W2']),
    ('Add(pre2, b2)', 'Y', lambda: env['pre2'] + env['b2']),
]

print('Forward Pass Trace')
print('=' * 60)
for desc, out_name, compute in steps:
    result = compute()
    env[out_name] = result
    print(f'  {desc:25s} → {out_name:8s} shape={str(result.shape):12s} '
          f'values={result.flatten()[:4].round(3)}')

print(f'\nORT result:    {ort_result.flatten().round(4)}')
print(f'Manual result: {env["Y"].flatten().round(4)}')
print(f'Match: {np.allclose(ort_result, env["Y"])}')

<a id='section-8'></a>
## Section 8: Static vs Dynamic Shapes

### Shape Specification in ONNX

| Specification | Meaning | Example |
|:---|:---|:---|
| Static integer | Fixed dimension | `[32, 4]` |
| Symbolic name | Named dynamic dim | `['batch', 4]` |
| `None` / unset | Unknown dimension | `[None, None]` |

### Trade-offs

| Aspect | Static Shapes | Dynamic Shapes |
|--------|:---:|:---:|
| Memory pre-allocation | Exact | Conservative |
| Kernel selection | Specialized | Generic |
| Operator fusion | More opportunities | Limited |
| Flexibility | Fixed batch only | Any batch size |
| File portability | Single configuration | Universal |

### Symbolic Dimension Sharing

When two inputs share a symbolic dimension name (e.g., both have dimension `'batch'`), the runtime knows those dimensions are always equal:

$$X: [\text{batch}, D], \quad Y: [\text{batch}, K] \implies X.\text{shape}[0] = Y.\text{shape}[0]$$

In [ ]:
# Demonstrate dynamic shapes
test_batches = [1, 4, 16, 64]

print('Dynamic shapes: same model, different batch sizes')
print('-' * 50)
for batch_size in test_batches:
    x_test = rng.standard_normal((batch_size, in_feat)).astype(np.float32)
    result = sess.run(None, {'X': x_test})[0]
    expected = np.maximum(x_test @ W1 + b1, 0) @ W2 + b2
    print(f'  batch={batch_size:3d}  input={str(x_test.shape):12s}  '
          f'output={str(result.shape):12s}  '
          f'correct={np.allclose(result, expected)}')

<a id='section-9'></a>
## Section 9: Key Takeaways & Interview Questions

### Summary

| Concept | Key Point |
|---------|----------|
| **DAG** | ONNX graphs are directed acyclic graphs: $G = (V, E)$ with no cycles |
| **Topological sort** | Guarantees a valid execution order: every input computed before use |
| **SSA naming** | Each tensor name produced exactly once; edges are implicit via name matching |
| **Forward pass** | Evaluate nodes in topological order, propagating values through the environment |
| **Components** | Inputs, outputs, nodes, initializers, and implicit edges |
| **Dynamic shapes** | Symbolic dimensions enable flexible batch sizes with the same model |

### Interview Questions

1. **Q**: What guarantees that an ONNX computation graph can be executed?
   - **A**: The acyclicity constraint ensures a topological ordering exists, meaning there is always a valid order to evaluate nodes such that all inputs are available before each node executes.

2. **Q**: How are edges represented in an ONNX graph?
   - **A**: Edges are implicit — they are inferred from SSA name matching. If node A produces output `"h"` and node B lists `"h"` as an input, there is an edge from A to B. There are no explicit edge objects.

3. **Q**: What is the computational complexity of the forward pass?
   - **A**: The graph traversal is $O(|V| + |E|)$, but total runtime is dominated by operator computations: $T = \sum_{v} T_{\text{op}}(v)$. For a model with a single MatMul$(N,D) \times (D,K)$, this is $O(NDK)$.

4. **Q**: What is the difference between static and dynamic shapes in ONNX?
   - **A**: Static shapes fix dimension sizes (e.g., `[32, 4]`), enabling aggressive optimizations but limiting flexibility. Dynamic shapes use `None` or symbolic names (e.g., `['batch', 4]`), allowing variable sizes at runtime but with fewer optimization opportunities.

---

**Next:** [Nodes, Edges, and Tensors](../02_Nodes_Edges_and_Tensors/) — Deep dive into ONNX wire format.